# Estudo Experimental: SAC vs TD3

Este notebook organiza o estudo sobre o impacto da exploração em algoritmos off-policy de Deep Reinforcement Learning. A implementação principal está nos scripts:

- `run_experiments.py`: executa os treinamentos SAC/TD3 e salva CSV/JSON.
- `generate_report.py`: gera gráficos, análise estatística e a apresentação `presentation.md`.

A execução completa recomendada para o trabalho final é `10 seeds x 100.000 passos`. Para validação rápida do pipeline, use `--quick`.

## 1. Preparação

Execute esta célula se estiver em um ambiente novo. No terminal local, a versão reprodutível usada foi uma virtualenv `.venv` com `requirements.txt`.

In [ ]:
# Opcional no Jupyter:
# %pip install -r requirements.txt

## 2. Configuração do Experimento

Parâmetros investigados:

| Algoritmo | Mecanismo de exploração | Valores |
|---|---|---|
| SAC | coeficiente de entropia alpha | 0.01, 0.05, 0.10, 0.20, auto |
| TD3 | desvio-padrão do ruído sigma | 0.05, 0.10, 0.20, 0.30 |

Ambiente principal: `Pendulum-v1`.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('project')
RESULTS_DIR = PROJECT_DIR / 'results'
FIGURES_DIR = PROJECT_DIR / 'figures'
PRESENTATION = Path('presentation.md')

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 3. Execução dos Experimentos

Para validar rapidamente o pipeline dentro do notebook, execute a célula com `--quick`.

Para a versão completa do trabalho, substitua por:

```bash
python run_experiments.py --timesteps 100000 --seeds 1-10 --eval-episodes 20
```

In [ ]:
# Smoke test reprodutível usado para gerar os resultados desta entrega.
!python run_experiments.py --quick

## 4. Geração da Apresentação Markdown

Esta etapa lê `project/results/results_all.csv` e `project/results/episode_series.json`, gera os gráficos e salva `presentation.md`.

In [ ]:
!python generate_report.py

## 5. Inspeção dos Resultados

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image

results = pd.read_csv(RESULTS_DIR / 'results_all.csv')
with open(RESULTS_DIR / 'metadata.json') as f:
    metadata = json.load(f)

print(metadata)
display(results.head())

In [ ]:
summary = (
    results
    .groupby(['algorithm', 'label', 'exploration'])
    .agg(mean_reward=('mean_reward', 'mean'), std_seeds=('mean_reward', 'std'), train_time_s=('train_time_s', 'mean'))
    .reset_index()
    .sort_values(['algorithm', 'label'])
)
summary['std_seeds'] = summary['std_seeds'].fillna(0)
display(summary)

## 6. Gráficos Gerados

In [ ]:
for fig in sorted(FIGURES_DIR.glob('*.png')):
    display(Markdown(f'### {fig.name}'))
    display(Image(filename=str(fig)))

## 7. Apresentação Final

O arquivo Markdown final fica em `presentation.md`.

In [ ]:
print(PRESENTATION.resolve())
display(Markdown(PRESENTATION.read_text(encoding='utf-8')[:4000]))